# Qrisp CUDA-Q Interface

Welcome to the Qrisp CUDA-Q interface tutorial! 🚀

This notebook will show you how to use Qrisp's new integration with [CUDA-Q](https://nvidia.github.io/cuda-quantum/), NVIDIA's high-performance quantum programming platform. You'll see how easy it is to run Qrisp programs on CUDA-Q, use hybrid quantum/classical logic, and leverage the full power of both Python and CUDA-Q in your quantum workflows.

Let's dive in and get quantum-y! ✨

## Hello World: Bell state

Let's start with the classic Bell state example. This demonstrates how to define a Qrisp kernel, run it with CUDA-Q, and see the results. Notice how natural it feels—just like writing regular Python!

In [36]:
import cudaq
from qrisp import *
from qrisp.jasp.cudaq_interface import qrisp_cudaq_kernel, FixedShapeNDArray

@qrisp_cudaq_kernel
def bell():
    qv = QuantumVariable(2)
    h(qv[0])
    cx(qv[0], qv[1])
    return measure(qv)

# Run the kernel once (statevector simulation)
result = bell()
print("Single run result:", result)

# Run the kernel multiple times (shots)
results = cudaq.run(bell, shots_count=10)
print("10 shots:", results)

Single run result: 3
10 shots: [0, 3, 0, 0, 3, 0, 3, 0, 0, 3]


## Complex Example: Hybrid Quantum-Classical Logic

Let's get a bit fancier! Here, we'll use Qrisp's ability to mix classical Python (like numpy and networkx) with quantum logic, and show off hybrid control flow—where quantum measurement results influence further quantum operations.

We'll also use Qrisp's operators module and trotterization for Hamiltonian simulation. This is a great way to see how expressive Qrisp kernels can be.

In [37]:
from qrisp.operators import X, Y, Z
import networkx as nx

@qrisp_cudaq_kernel
def hybrid_example():
    G = nx.Graph()
    G.add_nodes_from([0, 1, 2, 3])
    G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0)])

    H = sum(X(i) for i in G.nodes) + sum(Z(i)*Z(j) for i, j in G.edges)
    U = H.trotterization()

    a = QuantumFloat(4)
    b = QuantumFloat(4)

    a[:] = 5
    h(b)
    U(a, t=1.0, steps=10)

    # Real-time control flow based on measurement results
    c = measure(b) < 5
    with control(c):
        a += 1

    return measure(a)

result = hybrid_example()
print("Single run result:", result)

results = cudaq.run(hybrid_example, shots_count=10)
print("10 shots:", results)

Single run result: 11.0
10 shots: [0.0, 5.0, 5.0, 5.0, 1.0, 6.0, 6.0, 15.0, 5.0, 8.0]


### Extra Hybrid Control-Flow Example

Here is another hybrid pattern that is very common in practice: measure one qubit, and conditionally apply operations to a different qubits.

This is useful for adaptive protocols, feed-forward logic, and error-correction style workflows.

In [38]:
@qrisp_cudaq_kernel
def adaptive_flip_example():
    control_q = QuantumVariable(1)
    target_q = QuantumVariable(2)

    # Put control qubit into superposition, then branch on measurement.
    h(control_q[0])
    c = measure(control_q[0])

    with control(c == 1):
        x(target_q[0])
        z(target_q[1])

    return measure(target_q)

print("Adaptive hybrid control (10 shots):", cudaq.run(adaptive_flip_example, shots_count=10))

Adaptive hybrid control (10 shots): [1, 0, 0, 1, 1, 1, 0, 0, 0, 0]


## Parametrized Kernels

Kernel functions decorated with `@qrisp_cudaq_kernel` can accept classical parameters. The supported scalar parameter types are `int`, `float`, and `bool`. For array-valued parameters, use `FixedShapeNDArray(dtype, length)`, where `dtype` is one of those scalar types.

The example below shows a simple scalar-parametrized kernel — a single-qubit rotation by a runtime angle:

In [39]:
@qrisp_cudaq_kernel
def parametrized_kernel(angle: float):
    qv = QuantumFloat(1)
    ry(angle, qv[0])
    return measure(qv)

angle = 1.57
result_static = parametrized_kernel(angle)
print("Parametrized kernel (single run):", result_static)

Parametrized kernel (single run): 0.0


### Array Parameters: Static vs Dynamic Indexing

Array parameters are annotated with `FixedShapeNDArray(dtype, length)`. There are two important access patterns:

- **Static indexing**: the index is known at compile time (for example `angles[0]`).
- **Dynamic indexing**: the index is computed at runtime (for example from a measurement result like `angles[idx]`).

Both are supported, but dynamic indexing is more general and usually a bit more demanding for lowering/compilation. The examples below illustrate both patterns.

In [40]:
import numpy as np
import jax.numpy as jnp

# Static indexing example: access angles[0], where index is compile-time known.
@qrisp_cudaq_kernel
def static_index_kernel(angles: FixedShapeNDArray(float, 3)):
    qv = QuantumFloat(1)
    ry(angles[0], qv[0])
    return measure(qv[0])

angles_static = np.array([0.0, 1.57, 3.14])
result_static = static_index_kernel(angles_static)
print("Static indexing (single run):", result_static)

shots_static = cudaq.run(static_index_kernel, angles_static, shots_count=10)
print("Static indexing (10 shots):", shots_static)

Static indexing (single run): False
Static indexing (10 shots): [False, False, False, False, False, False, False, False, False, False]


In the previous cell, `angles[0]` is fixed at compile time.

Now let's switch to **dynamic indexing**: we compute the index at runtime from a measurement result and then read `angles[idx]`.

In [41]:
# Dynamic indexing example: index is computed at runtime.
@qrisp_cudaq_kernel
def dynamic_index_kernel(angles: FixedShapeNDArray(float, 3)):
    qv = QuantumVariable(1)
    idx = jnp.int32(measure(qv[0]))
    rx(angles[idx], qv[0])
    return measure(qv[0])

angles_dynamic = np.array([0.0, 1.57, 3.14])
result_dynamic = dynamic_index_kernel(angles_dynamic)
print("Dynamic indexing (single run):", result_dynamic)

shots_dynamic = cudaq.run(dynamic_index_kernel, angles_dynamic, shots_count=10)
print("Dynamic indexing (10 shots):", shots_dynamic)

Dynamic indexing (single run): False
Dynamic indexing (10 shots): [False, False, False, False, False, False, False, False, False, False]


## Static vs Dynamic Arrays in CUDA-Q Kernels

This topic is important, because array behavior changes depending on where the array comes from.

- **Static array**: created inside the kernel body (for example via `np.array([...])`).
- **Dynamic array parameter**: passed into the kernel and annotated as `FixedShapeNDArray(dtype, n)` or created inside the kernel (i.e., `jnp.array[...])`).

Practical rule of thumb:

- Static arrays defined inside the kernel support standard array transformations (for example `np.sin`).
- Dynamic array parameters are primarily meant for indexed access (`angles[i]`) and passing values into quantum operations. Transformations on dynamic arrays inside a kernel (for example `jnp.sin`) are not supported at this point.

The three examples below demonstrate these patterns in executable form.

In [42]:
# Example 1: static array created inside the kernel
@qrisp_cudaq_kernel
def static_array_kernel():
    qv = QuantumFloat(5)

    angles = np.array([1.57, 0.78, 0.39, 0.25, 0.12])
    angles = np.sin(angles)

    # Static array with static loop.
    for i in range(5):
        ry(angles[i], qv[i])

    return measure(qv)

print("Static array in-kernel (10 shots):", cudaq.run(static_array_kernel, shots_count=10))

Static array in-kernel (10 shots): [1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [43]:
# Example 2: dynamic array passed as kernel parameter
@qrisp_cudaq_kernel
def dynamic_array_kernel(angles: FixedShapeNDArray(float, 5)):
    qv = QuantumFloat(5)

    # Dynamic parameter array accessed element-wise.
    for i in jrange(5):
        ry(angles[i], qv[i])

    return measure(qv)

angles = np.array([1.57, 0.78, 0.39, 0.25, 0.12])
print("Dynamic array parameter (10 shots):", cudaq.run(dynamic_array_kernel, angles, shots_count=10))

Dynamic array parameter (10 shots): [1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0]


In [44]:
# Example 3: unsupported dynamic array operations inside the kernel (should fail to compile)
try:
    @qrisp_cudaq_kernel
    def dynamic_array_bad_kernel():
        qv = QuantumFloat(5)

        # Operations like jnp.sin on dynamic arrays are not currently supported inside the kernel.
        angles = jnp.array([1.57, 0.78, 0.39, 0.25, 0.12])
        angles = jnp.sin(angles)

        # Static array with static loop.
        for i in range(5):
            ry(angles[i], qv[i])

        return measure(qv)
except RuntimeError as e:
    print("Error defining kernel:", e)

Error defining kernel: Failed to compile Qrisp function 'dynamic_array_bad_kernel' to MLIR: Cannot lower this program to CUDA-Q because it still contains a ranked-tensor linalg.generic operation.

What this usually means:
- You performed arithmetic on traced jax.numpy arrays (for example array expressions inside the traced function), which introduced ranked tensor linalg ops.

How to fix it:
- Prefer scalar values inside the traced kernel body.
- Move array arithmetic outside tracing and pass only the needed elements/scalars into the kernel.
- If you pass arrays as kernel parameters, stick to simple indexed access patterns.

Location: loc(unknown)
Offending operation: GenericOp(
	%0 = linalg.generic {indexing_maps = [affine_map<(d0) -> (d0)>, affine_map<(d0) -> (d0)>], iterator_types = ["parallel"]} ins(%1 : tensor<5xf64>) outs(%2 : tensor<5xf64>) {
	^bb0(%arg5: f64, %arg6: f64):
	  %3 = math.sin %arg5 : f64
	  linalg.yield %3 : f64
	} -> tensor<5xf64>
)


## Advanced Example: Amplitude Amplification

Amplitude amplification is the generalization of Grover's algorithm. It boosts the probability of measuring a target state by repeatedly applying a state-preparation step and an oracle phase flip.

The kernel here takes an `int` parameter `iter` — the number of amplification iterations — which is a good example of an integer parametrized kernel.

The setup is:

- **State preparation**: a small `ry(π/8)` rotation initialises the qubit in a state with low initial overlap with `|True⟩`.
- **Oracle**: a `z` gate flips the phase of the `|True⟩` component.
- **Amplitude amplification**: each call to `amplitude_amplification(..., iter=iter)` applies `iter` Grover-style reflection pairs.

The example runs the kernel for `iter = 1, 2, 3` and prints the fraction of `True` outcomes over 100 shots. Each additional iteration boosts the success probability, illustrating how the number of iterations directly controls the amplification strength.

In [45]:
from qrisp.alg_primitives import amplitude_amplification

@qrisp_cudaq_kernel
def amplitude_amplification_example(iter: int):
    def state_function(qb):
        ry(np.pi / 8, qb)

    def oracle_function(qb):
        z(qb)

    qb = QuantumBool()
    state_function(qb)
    amplitude_amplification([qb], state_function, oracle_function, iter=iter)
    return measure(qb[0])

shots = 100
for it in [1, 2, 3]:
    result = cudaq.run(amplitude_amplification_example, it, shots_count=shots)
    ones_fraction = sum(result) / len(result)
    print(f"iter={it} -> fraction of 1 outcomes: {ones_fraction:.3f}")
    print(f"iter={it} -> raw shots: {result}\n")

/Users/renezander/Desktop/Qrisp/src/qrisp/jasp/mlir/quake_lowering/pass1_jasp_to_quake.py:873: UserWarning: Unsupported Jasp gate 'gphase' – removing jasp.quantum_gate from IR).
  _process_block(b)


iter=1 -> fraction of 1 outcomes: 0.310
iter=1 -> raw shots: [True, True, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, True, True, False, True, True, True, False, True, True, True, False, True, False, False, False, False, True, False, False, True, False, False, False, False, False, True, False, False, False, False, False, True, False, True, False, True, False, False, False, False, False, True, True, False, False, True, True, False, False, False, False, False, False, True, False, True, False, False, False, False, False, False, False, True, True, False, True, False, True, True, False, False, True, False, False, False, False, False, False, False, False, True]

iter=2 -> fraction of 1 outcomes: 0.680
iter=2 -> raw shots: [False, True, False, True, True, True, True, False, True, True, False, False, False, False, True, False, True, False, False, True, False, False, False, True, True, True, True, True, False, True, True, True, 

## Final Example: Custom MaxCut QAOA with CUDA-Q + SciPy

This final example puts everything together into a full hybrid variational loop.

- The kernel takes a parameter array `params = [gamma_0, gamma_1, beta_0, beta_1]`.
- The **cost operator** applies MaxCut phase separators along the graph edges.
- The **mixer** applies `rx` rotations to every qubit.
- A classical SciPy loop repeatedly calls `cudaq.run(...)` and optimizes the expected cut value computed from the measured bitstrings.

In [46]:
from scipy.optimize import minimize
import networkx as nx
import numpy as np

# 1. Define the MaxCut Instance
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (0, 2), (1, 3), (1,4), (2,5), (3,6), (4,5), (5,6), (6,4)]
N = 7
num_layers = 2

def apply_cost_operator(qv, gamma):
    for u, v in EDGES:
        cx(qv[u], qv[v])
        rz(2 * gamma, qv[v])
        cx(qv[u], qv[v])

def apply_mixer(qv, beta):
    for i in range(N):
        rx(2 * beta, qv[i])

# 2. Define the Qrisp CUDA-Q Kernel
@qrisp_cudaq_kernel
def variational_maxcut_kernel(params: FixedShapeNDArray(float, 2 * num_layers)):
    qv = QuantumVariable(N)
    
    # Start in the uniform superposition
    h(qv)

    num_layers = len(params) // 2
    
    # Apply multiple QAOA layers
    for i in jrange(num_layers):
        apply_cost_operator(qv, params[i])     # Gammas
        apply_mixer(qv, params[i + num_layers])         # Betas
        
    return measure(qv)

# 3. Simplified Classical Post-Processing
def calculate_cut_value(bitstring: int) -> int:
    """Calculates the cut size for a given measured integer bitstring."""
    cut = 0
    for u, v in EDGES:
        bit_u = (bitstring >> u) & 1
        bit_v = (bitstring >> v) & 1
        if bit_u != bit_v:
            cut += 1
    return cut

def objective_function(params: np.ndarray) -> float:
    """Runs the quantum kernel and evaluates the expected (average) cut."""
    # Sample the circuit
    samples = cudaq.run(variational_maxcut_kernel, params, shots_count=100)
    
    # Calculate the average cut value across all shots
    avg_cut = np.mean([calculate_cut_value(int(s)) for s in samples])
    
    # Scipy minimizes, so we return the negative expected cut
    return -avg_cut

# 4. Hybrid Optimization Loop
np.random.seed(42)
initial_params = np.random.uniform(0.0, np.pi, size=4)

print(f"Starting optimization... Initial expected cut: {-objective_function(initial_params):.3f}")

opt_result = minimize(
    objective_function,
    initial_params,
    method="COBYLA",
    options={"maxiter": 40}
)

# 5. Evaluate the Best Result
best_params = opt_result.x
final_samples = cudaq.run(variational_maxcut_kernel, best_params, shots_count=100)

# Find the best bitstring from our final heavily-sampled distribution
best_sample = max([int(s) for s in final_samples], key=calculate_cut_value)

print("\nOptimization finished! ✨")
print(f"Final expected cut: {-opt_result.fun:.3f}")
print(f"Best bitstring found (integer): {best_sample}")
print(f"Max cut value achieved: {calculate_cut_value(best_sample)}")
print(f"Optimized parameters: {np.round(best_params, 3)}")

Starting optimization... Initial expected cut: 5.430

Optimization finished! ✨
Final expected cut: 6.470
Best bitstring found (integer): 57
Max cut value achieved: 9
Optimized parameters: [1.04  2.803 1.54  1.29 ]


In [47]:
# You can combine all these features to build powerful, flexible quantum-classical workflows!

In [48]:
# For more, check out the Qrisp documentation and the test suite for inspiration!

In [49]:
# Happy quantum coding! 🚀

---

## Tips, Gotchas, and Debugging

- **Kernel parameters:** Use `FixedShapeNDArray` for numpy arrays, and regular Python types for scalars.
- **Hybrid control:** You can use `measure()` and `control()` to build hybrid quantum-classical logic.
- **Debugging:** If you get errors, check your kernel signature and make sure all types are supported by CUDA-Q.
- **More examples:** See the Qrisp test suite (`tests/jax_tests/test_qrisp_cudaq_kernel.py`) for more advanced patterns.
- **Limitations:** Some advanced Python features (closures, dynamic typing) may not be supported inside kernels—keep kernels simple and explicit for best results.

If you run into trouble, open an issue or check the Qrisp Discord for help!

---